# Long-read Quality Control


In [ ]:
# import necessary modules
import os
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
import numpy as np

sns.set_style("ticks",{'axes.grid' : True})
sns.set_palette("colorblind")

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["axes.titlesize"] = 30
plt.rcParams['axes.labelsize'] = 23.5
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['text.usetex'] = False
plt.rcParams["savefig.dpi"]=300


In [ ]:
input_nanopore_pre=list(snakemake.input.nanopore_pre)
input_nanopore_post=list(snakemake.input.nanopore_post)
input_pacbio=list(snakemake.input.pacbio)

SAMPLING=snakemake.params.sampling

output_summary_html=snakemake.output.summary_html
output_read_count_png=snakemake.output.read_count_png
output_read_count_svg=snakemake.output.read_count_svg
output_total_bases_png=snakemake.output.total_bases_png
output_total_bases_svg=snakemake.output.total_bases_svg
output_read_length_png=snakemake.output.read_length_png
output_read_length_svg=snakemake.output.read_length_svg
output_read_quality_png=snakemake.output.read_quality_png
output_read_quality_svg=snakemake.output.read_quality_svg


## Parse NanoStats


In [ ]:
def parse_nanostats(path, platform, stage):
    values = {}
    with open(path) as handle:
        for line in handle:
            line=line.strip()
            if len(line)==0:
                continue
            if ":" in line:
                key, value = line.split(":", 1)
            elif "\t" in line:
                key, value = line.split("\t", 1)
            else:
                continue
            key = key.strip()
            value = value.strip().replace(",", "")
            values[key] = value

    sample=os.path.basename(os.path.dirname(path))
    sample=sample.replace("_nanopore_preQC_" + SAMPLING, "")
    sample=sample.replace("_nanopore_postQC_" + SAMPLING, "")
    sample=sample.replace("pacbio_", "")
    sample=sample.replace("_" + SAMPLING, "")

    out = {
        "sample": sample,
        "platform": platform,
        "stage": stage,
        "path": path
    }

    key_map = {
        "Number of reads": "number_reads",
        "Total bases": "total_bases",
        "Median read length": "median_read_length",
        "Mean read length": "mean_read_length",
        "Read length N50": "read_length_N50",
        "Mean read quality": "mean_read_quality",
        "Median read quality": "median_read_quality"
    }

    for raw_key, clean_key in key_map.items():
        out[clean_key] = values.get(raw_key, np.nan)

    return out

rows=[]
for path in input_nanopore_pre:
    rows.append(parse_nanostats(path, "Nanopore", "Pre-QC"))
for path in input_nanopore_post:
    rows.append(parse_nanostats(path, "Nanopore", "Post-QC"))
for path in input_pacbio:
    rows.append(parse_nanostats(path, "PacBio HiFi", "Post-QC"))

qc_df=pd.DataFrame(rows)

numeric_cols=["number_reads", "total_bases", "median_read_length", "mean_read_length", "read_length_N50", "mean_read_quality", "median_read_quality"]
for col in numeric_cols:
    if col in qc_df.columns:
        qc_df[col]=pd.to_numeric(qc_df[col], errors="coerce")

qc_df["total_Mbp"]=qc_df["total_bases"]/1000000
qc_df=qc_df.sort_values(["platform", "sample", "stage"])
qc_df


In [ ]:
qc_df_out=qc_df.style.background_gradient(cmap="RdYlGn", subset=[col for col in ["number_reads", "total_Mbp", "read_length_N50", "mean_read_quality"] if col in qc_df.columns]).to_html()
with open(output_summary_html, "w") as fp:
    fp.write(qc_df_out)


## Number of reads


In [ ]:
fig_width=max(8, 0.5*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))
plot_df=qc_df.dropna(subset=["number_reads"]).copy()
plot_df["number_reads_million"]=plot_df["number_reads"]/1000000

sns.barplot(data=plot_df, x="sample", y="number_reads_million", hue="platform", ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Number of reads (million)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.savefig(output_read_count_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_read_count_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()


## Total bases


In [ ]:
fig_width=max(8, 0.5*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))
plot_df=qc_df.dropna(subset=["total_Mbp"]).copy()

sns.barplot(data=plot_df, x="sample", y="total_Mbp", hue="platform", ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Total bases (Mbp)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.savefig(output_total_bases_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_total_bases_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()


## Read length N50


In [ ]:
fig_width=max(8, 0.5*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))
plot_df=qc_df.dropna(subset=["read_length_N50"]).copy()
plot_df["read_length_N50_kbp"]=plot_df["read_length_N50"]/1000

sns.barplot(data=plot_df, x="sample", y="read_length_N50_kbp", hue="platform", ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Read length N50 (kbp)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.savefig(output_read_length_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_read_length_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()


## Mean read quality


In [ ]:
fig_width=max(8, 0.5*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))
plot_df=qc_df.dropna(subset=["mean_read_quality"]).copy()

sns.barplot(data=plot_df, x="sample", y="mean_read_quality", hue="platform", ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Mean read quality")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.savefig(output_read_quality_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_read_quality_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()
